# GPT od podstaw

Warsztaty w ramach Funduszu Zdolni (30 kwietnia - 2 maja 2026).

## Notebook 4: Samouwaga (self-attention)

**Inspiracja:** [Let's build GPT - Andrej Karpathy (YouTube)](https://www.youtube.com/watch?v=kCc8FmEb1nY) oraz [nanoGPT](https://github.com/karpathy/nanoGPT). Oryginalny artykuł: [Attention Is All You Need (2017)](https://arxiv.org/abs/1706.03762).

W [Notebooku 3](3_pytorch_network.ipynb) MLP patrzył na 3 znaki wstecz. Aby model widział dłuższy kontekst, zamiast sklejać embeddingi w jeden wektor, użyjemy **uwagi**: każdy token sam decyduje, na które poprzednie tokeny popatrzeć. To podstawowy element transformera.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from livelossplot import PlotLosses

torch.manual_seed(42)

### 1. Dane

In [ ]:
with open("data/pan-tadeusz.txt", "r", encoding="utf-8") as f:
    text = f.read()

chars = sorted(list(set(text)))
vocab_size = len(chars)
char2id = {c: i for i, c in enumerate(chars)}
id2char = {i: c for i, c in enumerate(chars)}

data = torch.tensor([char2id[c] for c in text], dtype=torch.long)
split_idx = int(len(data) * 0.9)
train_data = data[:split_idx]
test_data = data[split_idx:]

block_size = 16  # ile znaków wstecz model widzi
batch_size = 64

def losuj_paczke(split: str = "train") -> tuple[torch.Tensor, torch.Tensor]:
    zrodlo = train_data if split == "train" else test_data
    ix = torch.randint(0, len(zrodlo) - block_size - 1, (batch_size,))
    x = torch.stack([zrodlo[i:i+block_size] for i in ix])
    y = torch.stack([zrodlo[i+1:i+block_size+1] for i in ix])
    return x, y

print(f"Słownik: {vocab_size} znaków")

### 2. Głowa uwagi (Head)

Każdy token tworzy trzy wektory:
- **query (Q)** - czego szukam,
- **key (K)** - co oferuję,
- **value (V)** - co przekażę dalej.

Wagi to `softmax(Q · Kᵀ / √d)`. Maska trójkątna pilnuje, by token nie widział przyszłości. Wynik to średnia ważona wartości V.

In [ ]:
class Head(nn.Module):
    def __init__(self, n_embd: int, head_size: int, block_size: int):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wagi = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wagi = wagi.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wagi = F.softmax(wagi, dim=-1)
        return wagi @ v

### 3. Model z jedną głową uwagi

Embedding tokenu + embedding pozycji (sama uwaga jest "bez kolejności", więc trzeba jej powiedzieć, gdzie który token jest) → głowa uwagi → warstwa wyjściowa.

In [ ]:
class JednoglowyModel(nn.Module):
    def __init__(self, vocab_size: int, n_embd: int, head_size: int, block_size: int):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.head = Head(n_embd, head_size, block_size)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        x = self.token_emb(idx) + self.pos_emb(torch.arange(T))
        x = self.head(x)
        return self.lm_head(x)

    @torch.no_grad()
    def generuj(self, idx: torch.Tensor, max_nowych: int, temperatura: float = 1.0) -> torch.Tensor:
        for _ in range(max_nowych):
            idx_cond = idx[:, -self.block_size:]
            logits = self(idx_cond)[:, -1, :] / temperatura
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_idx], dim=1)
        return idx

In [ ]:
model = JednoglowyModel(vocab_size, n_embd=32, head_size=32, block_size=block_size)
print(f"Liczba parametrów: {sum(p.numel() for p in model.parameters()):,}")

### 4. Trening

Wyjście modelu ma kształt `(B, T, vocab_size)` - przewidujemy następny znak na każdej pozycji jednocześnie. Dlatego spłaszczamy do `(B*T, vocab_size)` przed cross-entropy.

In [ ]:
def oblicz_strate(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    logits = model(x)
    B, T, V = logits.shape
    return F.cross_entropy(logits.view(B*T, V), y.view(B*T))

@torch.no_grad()
def strata_test() -> float:
    model.eval()
    s = sum(oblicz_strate(*losuj_paczke("test")).item() for _ in range(20)) / 20
    model.train()
    return s

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
liveloss = PlotLosses()

for it in range(2000):
    xb, yb = losuj_paczke("train")
    loss = oblicz_strate(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if it % 50 == 0:
        liveloss.update({"loss": loss.item(), "val_loss": strata_test()})
        liveloss.send()

### 5. Generowanie

In [ ]:
def generuj_tekst(start_str: str, dlugosc: int = 300, temperatura: float = 1.0) -> str:
    idx = torch.tensor([[char2id[c] for c in start_str]], dtype=torch.long)
    out = model.generuj(idx, max_nowych=dlugosc, temperatura=temperatura)
    return "".join(id2char[i.item()] for i in out[0])

print(generuj_tekst("Litwo, ojczyzno moja", dlugosc=400, temperatura=0.8))

Jedna głowa to za mało. W [Notebooku 5](5_mini_gpt.ipynb) złożymy z nich pełny mini-GPT: wiele głów uwagi, warstwy feed-forward, residualne połączenia, LayerNorm i kilka bloków jeden na drugim.